# Experiment menu: each section produces one finding

> Runtime -> T4 or L4 GPU. Sections are independent — run the ones you want.
> Training sections take 10-40 min each on a T4; inference sections are quick.

Every experiment here maps to a claim you could put in a resume bullet or
defend in an interview. Run it, read the number, keep the table.

In [ ]:
%pip install -q unsloth

In [ ]:
!git clone https://github.com/zyziyun/qlora-lab.git
%cd /content/qlora-lab
!python scripts/make_data.py --n 800
import sys; sys.path.insert(0, 'src')

In [ ]:
# Shared helpers: load an adapter (or base), eval it on a dataset, free VRAM.
import time, torch, gc
from unsloth import FastLanguageModel
from qlora_lab import synth, train, dpo, agent, evaluate as qev, dataset as qds
from qlora_lab.dataset import SYSTEM_PROMPT
from qlora_lab.predict import Prediction
from qlora_lab.schema import Ticket, parse_ticket

test = qds.read_jsonl('data/test.jsonl')
ood  = synth.gen_ood()           # held-out out-of-distribution set

def run_eval(model, tokenizer, data, max_new_tokens=96):
    preds = []
    for e in data:
        msgs = [{'role':'system','content':SYSTEM_PROMPT},{'role':'user','content':e['message']}]
        ids = tokenizer.apply_chat_template(msgs, add_generation_prompt=True,
              return_tensors='pt', enable_thinking=False).to(model.device)
        t0=time.perf_counter(); out=model.generate(input_ids=ids, max_new_tokens=max_new_tokens,
              do_sample=False, pad_token_id=tokenizer.eos_token_id); dt=time.perf_counter()-t0
        g=out[0][ids.shape[1]:]
        preds.append(Prediction(raw=tokenizer.decode(g, skip_special_tokens=True),
                     latency_s=dt, prompt_tokens=int(ids.shape[1]), completion_tokens=int(g.shape[0])))
    return preds

def free(*objs):
    for o in objs:
        del o
    gc.collect(); torch.cuda.empty_cache()

def eval_adapter(path, data, **kw):
    m, t = FastLanguageModel.from_pretrained(path, max_seq_length=2048, load_in_4bit=True)
    FastLanguageModel.for_inference(m)
    rep = qev.evaluate(run_eval(m, t, data, **kw), data, in_price=0.05e-6, out_price=0.20e-6)
    free(m, t); return rep

## A. OOD gap, and whether data diversity closes it (experiment #3)
The shipped 1.7B adapter aces the in-distribution test but slips on unseen
styles. Retrain with `diverse=True` data and re-check the same OOD set.

In [ ]:
# in-distribution vs OOD for the shipped adapter
print('1.7B adapter, in-dist:', eval_adapter('outputs/adapter-1.7b', test).summary())
print('1.7B adapter, OOD    :', eval_adapter('outputs/adapter-1.7b', ood).summary())

In [ ]:
# train a diverse-data adapter and re-check OOD (~10 min on T4)
div_examples = synth.gen(800, seed=7, diverse=True)
parts = qds.split(div_examples, n_test=100, n_val=100)
tr, _ = qds.decontaminate(parts['train'], parts['test'])
qds.write_jsonl([qds.to_chat(e) for e in tr], 'data/train_diverse.jsonl')
train.train('data/train_diverse.jsonl', train.TrainConfig(
    base_model='unsloth/Qwen3-1.7B-bnb-4bit', output_dir='outputs/adapter-1.7b-diverse'))
print('diverse adapter, OOD :', eval_adapter('outputs/adapter-1.7b-diverse', ood).summary())
# Closed the gap -> diversity beats a bigger model. Still short -> 1.7B capacity ceiling.

## B. Guardrail: catch hallucinated order ids, measure escalation (experiment #2)
A deterministic check (order_id must be a substring of the message) catches the
dangerous failure the OOD set exposes, and routes only those to a strong model.

In [ ]:
m, t = FastLanguageModel.from_pretrained('outputs/adapter-1.7b', max_seq_length=2048, load_in_4bit=True)
FastLanguageModel.for_inference(m)
caught = 0
for e in ood:
    p = run_eval(m, t, [e])[0]
    tk, _ = parse_ticket(p.raw)
    if tk and not agent.order_id_in_message(tk, e['message']):
        caught += 1
        print('GUARD CAUGHT:', repr(e['message'][:60]), '-> claimed', tk.order_id)
print(f'\nwould escalate {caught}/{len(ood)} = {caught/len(ood):.0%} to the strong model')
print('at a 20x price gap, routing cost vs all-strong =', round((len(ood)+caught*20)/(len(ood)*20), 2))
free(m, t)

## C. Data scaling curve (experiment #4)
How many labeled examples does this narrow task actually need? Train on
increasing slices, eval each on the same test set.

In [ ]:
import json
full = qds.read_jsonl('data/train.jsonl')
rows = []
for k in [50, 150, 300, len(full)]:
    qds.write_jsonl(full[:k], f'data/train_{k}.jsonl')
    train.train(f'data/train_{k}.jsonl', train.TrainConfig(
        base_model='unsloth/Qwen3-1.7B-bnb-4bit', epochs=1, output_dir=f'outputs/scale-{k}'))
    r = eval_adapter(f'outputs/scale-{k}', test)
    rows.append((k, r.schema_validity, r.field_accuracy['priority'], r.exact_match))
    print(f'n={k:>4}  validity={r.schema_validity:.3f}  priority={r.field_accuracy["priority"]:.3f}  exact={r.exact_match:.3f}')
print('\nwhere the curve flattens is how much data you actually needed')

## D. Rank ablation (experiment #6)
Is r8 really within 1-2% of r64 on your task? Verify, do not assume.

In [ ]:
for r in [8, 16, 64]:
    train.train('data/train.jsonl', train.TrainConfig(
        base_model='unsloth/Qwen3-1.7B-bnb-4bit', lora_r=r, lora_alpha=2*r, epochs=1,
        output_dir=f'outputs/rank-{r}'))
    rep = eval_adapter(f'outputs/rank-{r}', test)
    print(f'r={r:>2}  validity={rep.schema_validity:.3f}  exact={rep.exact_match:.3f}  priority={rep.field_accuracy["priority"]:.3f}')

## E. Multi-base benchmark (experiment #7)
The resume's *benchmarking Qwen3, LLaMA3, Gemma* — same data, swap the base,
pick the Pareto point. (Each base downloads a few GB the first time.)

In [ ]:
for base in ['unsloth/Qwen3-1.7B-bnb-4bit', 'unsloth/Llama-3.2-3B-bnb-4bit']:
    tag = base.split('/')[-1]
    train.train('data/train.jsonl', train.TrainConfig(base_model=base, epochs=1, output_dir=f'outputs/{tag}'))
    rep = eval_adapter(f'outputs/{tag}', test)
    print(f'{tag:>26}  validity={rep.schema_validity:.3f}  exact={rep.exact_match:.3f}  out_tok={rep.mean_completion_tokens:.0f}')

## F. Detailed summaries (experiment #5)
The dual of 'the model learns your data's shape': feed summaries that carry the
order id and key fact, and the tuned model emits detailed summaries too.

In [ ]:
det = synth.gen(800, seed=7, detailed_summary=True)
p = qds.split(det, n_test=100, n_val=100); tr,_ = qds.decontaminate(p['train'], p['test'])
qds.write_jsonl([qds.to_chat(e) for e in tr], 'data/train_detailed.jsonl')
train.train('data/train_detailed.jsonl', train.TrainConfig(
    base_model='unsloth/Qwen3-1.7B-bnb-4bit', epochs=1, output_dir='outputs/adapter-detailed'))
m,t = FastLanguageModel.from_pretrained('outputs/adapter-detailed', max_seq_length=2048, load_in_4bit=True)
FastLanguageModel.for_inference(m)
print(run_eval(m, t, [p['test'][0]])[0].raw)
free(m, t)

## G. DPO second stage (experiment #9)
SFT makes it correct; DPO makes it prefer the compact prose-free style over a
chatty fenced one. Watch output tokens drop with no loss of validity.

In [ ]:
# preference pairs: chosen = compact JSON, rejected = chatty fenced JSON
import json
pairs = dpo.make_preference_pairs(qds.read_jsonl('data/test.jsonl')[:200] if False else synth.gen(400, seed=11))
with open('data/prefs.jsonl','w') as f:
    for r in pairs: f.write(json.dumps(r)+'\n')
dpo.train_dpo('data/prefs.jsonl', base_model='unsloth/Qwen3-1.7B-bnb-4bit',
    cfg=dpo.DPOConfig(sft_adapter='outputs/adapter-1.7b', output_dir='outputs/adapter-dpo'))
print('SFT  :', eval_adapter('outputs/adapter-1.7b', test).summary())
print('DPO  :', eval_adapter('outputs/adapter-dpo', test).summary())
# expect similar validity, fewer output tokens, zero preambles

## H. vLLM concurrency (experiment #8)
Lives in `colab_serve_vllm.ipynb` (needs a running server). The snippet: send N
requests through a ThreadPoolExecutor and compare throughput to serial — the
gap is vLLM's continuous batching + PagedAttention, the S4 goodput story made real.
```python
from concurrent.futures import ThreadPoolExecutor
with ThreadPoolExecutor(max_workers=30) as ex:
    out = list(ex.map(lambda e: predict.extract(client,'ticket',e['message'],extra_body=NO_THINK), test))
```